Notebook 7B — Tuned, Unpruned 61-Predictor XGBoost

This notebook completes the baseline-symmetry analysis by adding a tuned, unpruned XGBoost configuration. It uses the same development/test partition, Optuna sampler, 50-trial budget, five-fold stratified cross-validation, ROC-AUC objective, seed, and search space used for SP-XGBoost in Notebook 7.

The four configurations are:

1. Baseline XGBoost: 61 predictors, untuned.
2. S-XGBoost: 49 predictors, untuned.
3. Tuned Baseline XGBoost: 61 predictors, tuned.
4. SP-XGBoost: 49 predictors, tuned.

The independent test set is not used for tuning. Because this supplementary analysis was added after earlier test results had been reviewed, its test-set contrasts are reported as post-hoc component comparisons.


In [1]:
# CELL 1 — INITIALISATION
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import optuna
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold

PRIMARY_SEED = 42
N_SPLITS = 5
N_TRIALS = 50
TARGET = "RISK_BINARY"

PROJECT_ROOT = Path("..")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
SYMMETRY_DIR = RESULTS_DIR / "baseline_symmetry"

for directory in [MODELS_DIR, RESULTS_DIR, SYMMETRY_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("NOTEBOOK 7B — TUNED, UNPRUNED 61-PREDICTOR XGBOOST")
print("=" * 70)
print(f"XGBoost: {xgb.__version__}")
print(f"Optuna:  {optuna.__version__}")


NOTEBOOK 7B — TUNED, UNPRUNED 61-PREDICTOR XGBOOST
XGBoost: 3.3.0
Optuna:  4.9.0


In [2]:
# CELL 2 — LOAD LOCKED 61-PREDICTOR DATA
train_df = pd.read_csv(PROCESSED_DIR / "train_binary_model.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test_binary_model.csv")

X_train_61 = train_df.drop(columns=[TARGET]).copy()
y_train = train_df[TARGET].astype(int).copy()
X_test_61 = test_df.drop(columns=[TARGET]).copy()
y_test = test_df[TARGET].astype(int).copy()

assert X_train_61.shape == (468, 61)
assert X_test_61.shape == (117, 61)
assert y_train.shape == (468,)
assert y_test.shape == (117,)
assert list(X_train_61.columns) == list(X_test_61.columns)
assert X_train_61.select_dtypes(exclude=np.number).shape[1] == 0
assert X_test_61.select_dtypes(exclude=np.number).shape[1] == 0
assert X_train_61.isna().sum().sum() == 0
assert X_test_61.isna().sum().sum() == 0
assert y_train.value_counts().sort_index().to_dict() == {0: 244, 1: 224}
assert y_test.value_counts().sort_index().to_dict() == {0: 61, 1: 56}

print("Training matrix:", X_train_61.shape)
print("Testing matrix: ", X_test_61.shape)
print("✓ Locked 61-predictor matrices validated")


Training matrix: (468, 61)
Testing matrix:  (117, 61)
✓ Locked 61-predictor matrices validated


In [3]:
# CELL 3 — LOCK MATCHED CROSS-VALIDATION
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=PRIMARY_SEED,
)

# Materialise fold indices so the exact folds are auditable and reusable.
cv_splits = list(cv.split(X_train_61, y_train))
assert len(cv_splits) == 5

fold_rows = []
for fold_number, (train_idx, valid_idx) in enumerate(cv_splits, start=1):
    fold_rows.extend(
        {"Row_Index": int(idx), "Fold": fold_number, "Role": "train"}
        for idx in train_idx
    )
    fold_rows.extend(
        {"Row_Index": int(idx), "Fold": fold_number, "Role": "validation"}
        for idx in valid_idx
    )

fold_path = SYMMETRY_DIR / "tuned_baseline_cv_fold_indices.csv"
pd.DataFrame(fold_rows).to_csv(fold_path, index=False)

print("CV: StratifiedKFold(n_splits=5, shuffle=True, random_state=42)")
print("✓ Exact fold indices saved")
print("✓ Independent test set excluded from CV")


CV: StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
✓ Exact fold indices saved
✓ Independent test set excluded from CV


In [4]:
# CELL 4 — IDENTICAL OPTUNA SEARCH SPACE
def suggest_xgb_params(trial):
    return {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.30, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.60, 1.00),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.60, 1.00),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "random_state": PRIMARY_SEED,
        "n_jobs": -1,
    }


def objective(trial):
    params = suggest_xgb_params(trial)
    fold_scores = []

    for train_idx, valid_idx in cv_splits:
        model = xgb.XGBClassifier(**params)
        model.fit(
            X_train_61.iloc[train_idx],
            y_train.iloc[train_idx],
            verbose=False,
        )
        valid_prob = model.predict_proba(X_train_61.iloc[valid_idx])[:, 1]
        fold_scores.append(
            roc_auc_score(y_train.iloc[valid_idx], valid_prob)
        )

    return float(np.mean(fold_scores))

print("✓ Search space matches Notebook 7")
print("✓ Objective: mean five-fold validation ROC-AUC")


✓ Search space matches Notebook 7
✓ Objective: mean five-fold validation ROC-AUC


In [5]:
# CELL 5 — RUN 50-TRIAL OPTUNA STUDY
sampler = optuna.samplers.TPESampler(seed=PRIMARY_SEED)
study_61 = optuna.create_study(
    direction="maximize",
    sampler=sampler,
    study_name="Tuned_Unpruned_61_XGBoost_ROC_AUC",
)

start = perf_counter()
study_61.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
optuna_runtime_seconds = perf_counter() - start

assert len(study_61.trials) == 50
assert np.isfinite(study_61.best_value)

print(f"Completed trials: {len(study_61.trials)}")
print(f"Best trial:      {study_61.best_trial.number}")
print(f"Best CV ROC-AUC: {study_61.best_value:.6f}")
print(f"Optuna runtime:  {optuna_runtime_seconds:.3f} seconds")
print("Best parameters:")
for name, value in study_61.best_params.items():
    print(f"  {name}: {value}")


[I 2026-09-08 01:10:21,194] A new study created in memory with name: Tuned_Unpruned_61_XGBoost_ROC_AUC


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-09-08 01:10:22,084] Trial 0 finished with value: 0.683651309008452 and parameters: {'n_estimators': 250, 'max_depth': 8, 'learning_rate': 0.1205712628744377, 'min_child_weight': 6, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'gamma': 0.2904180608409973, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.2537815508265665}. Best is trial 0 with value: 0.683651309008452.
[I 2026-09-08 01:10:23,290] Trial 1 finished with value: 0.6429042121899264 and parameters: {'n_estimators': 400, 'max_depth': 2, 'learning_rate': 0.2708160864249968, 'min_child_weight': 9, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'gamma': 0.9170225492671691, 'reg_alpha': 5.472429642032198e-06, 'reg_lambda': 0.12561043700013558}. Best is trial 0 with value: 0.683651309008452.
[I 2026-09-08 01:10:24,002] Trial 2 finished with value: 0.7154974919260633 and parameters: {'n_estimators': 250, 'max_depth': 4, 'learning_rate': 0.08012737503998542, 'min_child_weigh

In [6]:
# CELL 6 — SAVE OPTUNA PROVENANCE
trials_path = SYMMETRY_DIR / "tuned_baseline_61_optuna_trials.csv"
params_path = SYMMETRY_DIR / "tuned_baseline_61_best_params.csv"

study_61.trials_dataframe().to_csv(trials_path, index=False)
pd.DataFrame([study_61.best_params]).to_csv(params_path, index=False)

assert trials_path.exists()
assert params_path.exists()
assert len(pd.read_csv(trials_path)) == 50
print("✓ Trial history and best parameters saved")


✓ Trial history and best parameters saved


In [7]:
# CELL 7 — REFIT TUNED BASELINE ON ALL DEVELOPMENT RECORDS
final_params_61 = study_61.best_params.copy()
final_params_61.update({
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": PRIMARY_SEED,
    "n_jobs": -1,
})

tuned_baseline_xgb = xgb.XGBClassifier(**final_params_61)

start = perf_counter()
tuned_baseline_xgb.fit(X_train_61, y_train, verbose=False)
fit_runtime_seconds = perf_counter() - start

assert tuned_baseline_xgb.n_features_in_ == 61
print(f"Final fit runtime: {fit_runtime_seconds:.6f} seconds")
print("✓ Tuned unpruned model fitted on 468 development records")


Final fit runtime: 0.083780 seconds
✓ Tuned unpruned model fitted on 468 development records


In [8]:
# CELL 8 — ONE INDEPENDENT-TEST EVALUATION
start = perf_counter()
tuned_61_pred = tuned_baseline_xgb.predict(X_test_61)
tuned_61_prob = tuned_baseline_xgb.predict_proba(X_test_61)[:, 1]
inference_runtime_seconds = perf_counter() - start

tn, fp, fn, tp = confusion_matrix(y_test, tuned_61_pred).ravel()

tuned_61_metrics = {
    "Model": "Tuned Baseline XGBoost",
    "Predictors": 61,
    "Pruned": "No",
    "Tuned": "Yes",
    "ROC_AUC": roc_auc_score(y_test, tuned_61_prob),
    "PR_AUC": average_precision_score(y_test, tuned_61_prob),
    "F1": f1_score(y_test, tuned_61_pred, zero_division=0),
    "Precision": precision_score(y_test, tuned_61_pred, zero_division=0),
    "Recall": recall_score(y_test, tuned_61_pred, zero_division=0),
    "Specificity": tn / (tn + fp),
    "Accuracy": accuracy_score(y_test, tuned_61_pred),
    "TN": int(tn),
    "FP": int(fp),
    "FN": int(fn),
    "TP": int(tp),
}

print(pd.DataFrame([tuned_61_metrics]).to_string(index=False))
print(f"Combined test inference runtime: {inference_runtime_seconds:.6f} seconds")


                 Model  Predictors Pruned Tuned  ROC_AUC   PR_AUC       F1  Precision   Recall  Specificity  Accuracy  TN  FP  FN  TP
Tuned Baseline XGBoost          61     No   Yes  0.81089 0.798013 0.686869   0.790698 0.607143     0.852459  0.735043  52   9  22  34
Combined test inference runtime: 0.031395 seconds


In [9]:
# CELL 9 — SAVE MODEL, PREDICTIONS, METRICS, AND RUNTIME
model_path = MODELS_DIR / "tuned_baseline_xgboost_61.pkl"
prediction_path = SYMMETRY_DIR / "tuned_baseline_61_test_predictions.csv"
metric_path = SYMMETRY_DIR / "tuned_baseline_61_test_metrics.csv"
runtime_path = SYMMETRY_DIR / "tuned_baseline_61_runtime.csv"

joblib.dump(tuned_baseline_xgb, model_path)

pd.DataFrame({
    "Test_Row": np.arange(1, len(y_test) + 1),
    "RISK_BINARY_ACTUAL": y_test.to_numpy(),
    "Tuned_Baseline_Probability": tuned_61_prob,
    "Tuned_Baseline_Prediction": tuned_61_pred,
}).to_csv(prediction_path, index=False)

pd.DataFrame([tuned_61_metrics]).to_csv(metric_path, index=False)

pd.DataFrame([{
    "Optuna_50_Trial_5Fold_Seconds": optuna_runtime_seconds,
    "Final_Fit_468x61_Seconds": fit_runtime_seconds,
    "Combined_Predict_And_PredictProba_117x61_Seconds": inference_runtime_seconds,
    "Mean_Inference_Per_Student_Seconds": inference_runtime_seconds / len(X_test_61),
}]).to_csv(runtime_path, index=False)

for path in [model_path, prediction_path, metric_path, runtime_path]:
    assert path.exists(), path

reloaded_model = joblib.load(model_path)
assert np.array_equal(reloaded_model.predict(X_test_61), tuned_61_pred)
assert np.allclose(reloaded_model.predict_proba(X_test_61)[:, 1], tuned_61_prob)
print("✓ Tuned baseline artifacts saved and model reload verified")


✓ Tuned baseline artifacts saved and model reload verified


In [10]:
# CELL 10 — CONSTRUCT COMPLETE 2×2 COMPARISON
baseline_path = RESULTS_DIR / "baseline_xgboost_binary_results.csv"
s_xgb_path = RESULTS_DIR / "s_xgboost_binary_results.csv"
three_model_path = RESULTS_DIR / "three_model_comparison.csv"

for path in [baseline_path, s_xgb_path, three_model_path]:
    assert path.exists(), f"Required prior result not found: {path}"

baseline_row = pd.read_csv(baseline_path).iloc[0]
s_row = pd.read_csv(s_xgb_path).iloc[0]
three_model_df = pd.read_csv(three_model_path)
sp_row = three_model_df.loc[three_model_df["Model"] == "SP-XGBoost"].iloc[0]

def standard_row(name, predictors, pruned, tuned, row):
    return {
        "Model": name,
        "Predictors": predictors,
        "Pruned": pruned,
        "Tuned": tuned,
        "ROC_AUC": float(row["ROC_AUC"]),
        "PR_AUC": float(row["PR_AUC"]),
        "F1": float(row["F1"]),
        "Precision": float(row["Precision"]),
        "Recall": float(row["Recall"]),
        "Specificity": float(row["Specificity"]),
        "Accuracy": float(row["Accuracy"]),
    }

comparison_2x2 = pd.DataFrame([
    standard_row("Baseline XGBoost", 61, "No", "No", baseline_row),
    standard_row("S-XGBoost", 49, "Yes", "No", s_row),
    standard_row("Tuned Baseline XGBoost", 61, "No", "Yes", pd.Series(tuned_61_metrics)),
    standard_row("SP-XGBoost", 49, "Yes", "Yes", sp_row),
])

comparison_path = SYMMETRY_DIR / "xgboost_full_2x2_comparison.csv"
comparison_2x2.to_csv(comparison_path, index=False)

print("=" * 100)
print("COMPLETE XGBOOST 2×2 COMPONENT COMPARISON")
print("=" * 100)
print(comparison_2x2.to_string(index=False))


COMPLETE XGBOOST 2×2 COMPONENT COMPARISON
                 Model  Predictors Pruned Tuned  ROC_AUC   PR_AUC       F1  Precision   Recall  Specificity  Accuracy
      Baseline XGBoost          61     No    No 0.756733 0.721970 0.629630   0.653846 0.607143     0.704918  0.658120
             S-XGBoost          49    Yes    No 0.768150 0.739462 0.647619   0.693878 0.607143     0.754098  0.683761
Tuned Baseline XGBoost          61     No   Yes 0.810890 0.798013 0.686869   0.790698 0.607143     0.852459  0.735043
            SP-XGBoost          49    Yes   Yes 0.813817 0.775148 0.705882   0.782609 0.642857     0.836066  0.743590


In [11]:
# CELL 11 — COMPONENT CONTRASTS
metric_columns = [
    "ROC_AUC", "PR_AUC", "F1", "Precision", "Recall", "Specificity", "Accuracy"
]
indexed = comparison_2x2.set_index("Model")

contrast_specs = {
    "Pruning effect without tuning (S − Baseline)": ("S-XGBoost", "Baseline XGBoost"),
    "Pruning effect after tuning (SP − Tuned Baseline)": ("SP-XGBoost", "Tuned Baseline XGBoost"),
    "Tuning effect without pruning (Tuned Baseline − Baseline)": ("Tuned Baseline XGBoost", "Baseline XGBoost"),
    "Tuning effect after pruning (SP − S)": ("SP-XGBoost", "S-XGBoost"),
}

contrast_rows = []
for contrast, (model_a, model_b) in contrast_specs.items():
    row = {"Contrast": contrast}
    for metric in metric_columns:
        row[f"Delta_{metric}"] = indexed.loc[model_a, metric] - indexed.loc[model_b, metric]
    contrast_rows.append(row)

component_contrasts = pd.DataFrame(contrast_rows)
contrast_path = SYMMETRY_DIR / "xgboost_2x2_component_contrasts.csv"
component_contrasts.to_csv(contrast_path, index=False)

print(component_contrasts.to_string(index=False))
print("\nInterpretation safeguard:")
print("These are post-hoc descriptive component contrasts, not causal effects.")


                                                 Contrast  Delta_ROC_AUC  Delta_PR_AUC  Delta_F1  Delta_Precision  Delta_Recall  Delta_Specificity  Delta_Accuracy
             Pruning effect without tuning (S − Baseline)       0.011417      0.017492  0.017989         0.040031      0.000000           0.049180        0.025641
        Pruning effect after tuning (SP − Tuned Baseline)       0.002927     -0.022866  0.019014        -0.008089      0.035714          -0.016393        0.008547
Tuning effect without pruning (Tuned Baseline − Baseline)       0.054157      0.076044  0.057239         0.136852      0.000000           0.147541        0.076923
                     Tuning effect after pruning (SP − S)       0.045667      0.035686  0.058263         0.088731      0.035714           0.081967        0.059829

Interpretation safeguard:
These are post-hoc descriptive component contrasts, not causal effects.


In [12]:
# CELL 12 — FINAL VALIDATION
assert X_train_61.shape == (468, 61)
assert X_test_61.shape == (117, 61)
assert len(study_61.trials) == 50
assert comparison_2x2.shape[0] == 4
assert set(comparison_2x2["Predictors"]) == {49, 61}
assert set(comparison_2x2["Pruned"]) == {"Yes", "No"}
assert set(comparison_2x2["Tuned"]) == {"Yes", "No"}
assert len(tuned_61_pred) == 117
assert len(tuned_61_prob) == 117
assert np.all((tuned_61_prob >= 0) & (tuned_61_prob <= 1))
assert pd.read_csv(prediction_path).shape == (117, 4)
assert pd.read_csv(comparison_path).shape[0] == 4
assert pd.read_csv(contrast_path).shape[0] == 4

print("=" * 70)
print("NOTEBOOK 7B — FINAL VALIDATION")
print("=" * 70)
print("✓ 61-predictor unpruned input validated")
print("✓ Same Optuna search space and 50-trial budget applied")
print("✓ Same five stratified folds and seed 42 applied")
print("✓ Independent test excluded from tuning")
print("✓ Tuned unpruned model fitted and saved")
print("✓ 117 paired test predictions saved")
print("✓ Complete four-configuration comparison saved")
print("✓ Four component contrasts saved")
print("✓ Baseline-symmetry analysis completed")


NOTEBOOK 7B — FINAL VALIDATION
✓ 61-predictor unpruned input validated
✓ Same Optuna search space and 50-trial budget applied
✓ Same five stratified folds and seed 42 applied
✓ Independent test excluded from tuning
✓ Tuned unpruned model fitted and saved
✓ 117 paired test predictions saved
✓ Complete four-configuration comparison saved
✓ Four component contrasts saved
✓ Baseline-symmetry analysis completed
